In [1]:
# Install Hugging Face transformers if not already
!pip install transformers

In [2]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

In [3]:
# 1. Load pretrained BERT + official vocab
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

mask_token = tokenizer.mask_token   # "[MASK]"
mask_token_id = tokenizer.mask_token_id
print("Mask token:", mask_token, "| ID:", mask_token_id)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mask token: [MASK] | ID: 103


In [4]:
# 2. Function to predict masked token(s)
def predict_masked_tokens(text, top_k=5):
    # Encode text
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    # Ensure [MASK] exists
    if mask_token_id not in input_ids:
        return f"⚠️ No {mask_token} token found in: {text}"

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = outputs.logits

    # Find mask positions
    mask_positions = (input_ids == mask_token_id).nonzero(as_tuple=True)[1]

    results = []
    for pos in mask_positions:
        probs = predictions[0, pos].softmax(dim=0)
        top_tokens = torch.topk(probs, top_k)
        tokens = [tokenizer.decode([i]) for i in top_tokens.indices]
        scores = [round(float(s), 4) for s in top_tokens.values]
        results.append(list(zip(tokens, scores)))

    return results


In [8]:

examples = [
    "Sun rises in the  [MASK].",
    " I went to [MASK] for studying.",
    "Football Is a good [MASK] ."
]

for ex in examples:
    print(f"\nInput: {ex}")
    print("Predictions:", predict_masked_tokens(ex))


Input: Sun rises in the  [MASK].
Predictions: [[('sky', 0.7093), ('west', 0.0882), ('east', 0.0741), ('morning', 0.0238), ('distance', 0.0122)]]

Input:  I went to [MASK] for studying.
Predictions: [[('school', 0.0951), ('london', 0.0934), ('college', 0.0364), ('paris', 0.0362), ('england', 0.0246)]]

Input: Football Is a good [MASK] .
Predictions: [[('sport', 0.7786), ('thing', 0.1361), ('game', 0.0436), ('place', 0.0026), ('life', 0.0022)]]
